In [1]:
import json
import numpy as np
import pandas as pd

import os 

from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
import os
print(os.getcwd())
print(os.listdir())

/Users/sanshwehtet/Documents/Year_3/Sem1/ML/PIMA Dataset/notebooks
['diabetes_final_proj.ipynb', 'check_export.ipynb', 'diabetes_final_proj.html']


In [4]:
model = keras.models.load_model("../app/model.keras")
with open("../app/preprocessing.json") as f:
    prep_info = json.load(f)

saved = pd.read_csv("../app/test_prediction.csv")

print(model.summary())
print(prep_info.keys())
print(saved["probability"])

Model: "sequential_51"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ relu_layer_1 (Dense)            │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_39 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284 (5.02 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 643 (2.52 KB)

None
dict_keys(['feature_order', 'zero_as_missing', 'medians', 'mean', 'std'])
0      0.675358
1      0.077248
2      0.352199
3      0.365456
4      0.010609
         ...   
149    0.003213
150    0.056072
151    0.102990
152    0.906057
153    0.013416
Name: probability, Length: 154, dtype: float64


In [5]:
dataset_url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/refs/heads/master/diabetes.csv"            # copy the exact URL from your notebook
dataset_df = pd.read_csv(dataset_url, header=0)

print(dataset_df.shape)

(768, 9)


In [6]:
X = dataset_df.drop(columns=['Outcome']).values.astype('float32')
y = dataset_df['Outcome'].values.astype('float32')

print(X.shape, X.dtype)
print(y.shape)

(768, 8) float32
(768,)


In [7]:
print(type(prep_info))
print(prep_info.keys())

<class 'dict'>
dict_keys(['feature_order', 'zero_as_missing', 'medians', 'mean', 'std'])


In [8]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, 
    test_size = 0.2, 
    train_size = 0.8,
    stratify=y, 
    random_state=42
)

print(X_test.shape)

(154, 8)


In [9]:
# Column Order Check
print(list(dataset_df.drop(columns=["Outcome"]).columns) == prep_info["feature_order"])

True


In [10]:
COLUMN_NAMES = [item for item in dataset_df.columns]

# cols_with_zeros = [
#     'Glucose',
#     'BloodPressure',
#     'SkinThickness',
#     'Insulin',
#     'BMI'
# ]
# # feature col names
# feature_cols = [c for c in COLUMN_NAMES if c!= 'Outcome']
# # idx of col with zero in relation to exisiting feature col idxs
# zero_col_idx = [feature_cols.index(c) for c in cols_with_zeros]

print(COLUMN_NAMES)
print(prep_info["feature_order"])

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


In [11]:
print(prep_info.keys())

dict_keys(['feature_order', 'zero_as_missing', 'medians', 'mean', 'std'])


In [12]:
print(prep_info["medians"])

[3.0, 117.0, 72.0, 29.0, 125.0, 32.400001525878906, 0.3824999928474426, 29.0]


In [13]:
# taken from the prep_data(x):
def prep(X, prep_info = prep_info):
    X = X.copy()

    medians = np.array(prep_info["medians"], dtype = "float32")
    mean    = np.array(prep_info["mean"], dtype = "float32")
    std     = np.array(prep_info["std"], dtype = "float32")

    # zero --> Nan
    for col_name in prep_info["zero_as_missing"]:

        col_idx = prep_info["feature_order"].index(col_name)

        # 2. zeros -> NaN, only in the zero_as_missing columns
        for i in range(len(X)):
            if X[i, col_idx] == 0:
                X[i, col_idx] = np.nan
        
        # 3. Nan --> median
        for col_idx in range(X.shape[1]):
            missing = np.isnan(X[:, col_idx])
            X[missing, col_idx] = medians[col_idx]
    
    return ((X - mean)/ std).astype("float32")
    # print(medians)
        

In [14]:
X_test_prepped = prep(X_test, prep_info)

print(np.isnan(X_test_prepped).any())          # False, no missing values left
print(X_test_prepped.mean(axis=0).round(2))    # all close to 0
print(X_test[0])                                # the original row is still unchanged

False
[ 0.04 -0.    0.1   0.04  0.19  0.01 -0.08 -0.05]
[  7.    159.     64.      0.      0.     27.4     0.294  40.   ]


In [ ]:
new = model.predict(X_test_prepped).ravel() # unravel for flat dimension
old = saved["probability"].values

print(np.allclose(old, new, atol=0.00001))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
True
